In [1]:
# Cell 1 - Setup
# If needed, uncomment:
# !pip install -q pandas numpy transformers torch

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 13
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [2]:
# Cell 2 - Configuration
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
EVAL_FILE = Path("TestData.xlsx.csv")
TOP_K = 3
# Set to 5 for smoke-check mode, or None for full evaluation
N_EVAL_LIMIT = None
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_FILE = OUTPUT_DIR / "baseline_predictions.jsonl"
ADJUDICATION_QUEUE_FILE = OUTPUT_DIR / "baseline_adjudication_queue.csv"
ADJUDICATION_RESOLVED_FILE = OUTPUT_DIR / "baseline_adjudication_resolved.csv"
METRICS_FILE = OUTPUT_DIR / "baseline_metrics.json"
GEN_CONFIG = {
    "do_sample": False,
    "temperature": 0.0,
    "max_new_tokens": 256,
}


In [3]:
# Cell 3 - Load evaluation data (robust CSV parsing)
import csv

rows = []
with open(EVAL_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for row in reader:
        if not row:
            continue
        if len(row) < 3:
            continue

        rows.append({
            "Rank": row[0].strip(),
            "Film": row[1].strip(),
            "Description": ",".join(row[2:]).strip(),
        })

eval_df = pd.DataFrame(rows)

required_cols = {"Rank", "Film", "Description"}
missing = required_cols - set(eval_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

eval_df["example_id"] = [f"eval_{i:04d}" for i in range(1, len(eval_df) + 1)]
eval_df["prompt"] = eval_df["Description"].astype(str).str.strip()
eval_df["gold_title"] = eval_df["Film"].astype(str).str.strip()

eval_df = eval_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

if N_EVAL_LIMIT is not None:
    eval_df = eval_df.head(N_EVAL_LIMIT).copy()

eval_df = eval_df[["example_id", "Rank", "prompt", "gold_title"]]
display(eval_df.head(3))
print(f"Loaded eval rows: {len(eval_df)}")


,example_id,Rank,prompt,gold_title
0,eval_0038,38,A young heir leaves home after a family traged...,The Lion King
1,eval_0063,63,A traveler tries to mediate between industrial...,Princess Mononoke
2,eval_0084,84,A crew embarks on a mission with an onboard sy...,2001: A Space Odyssey


Loaded eval rows: 100


In [4]:
# Cell 4 - Build fixed catalog
def normalize_title(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace("’", "'").replace("`", "'")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*([:,!?.;])\s*", r"\1", text)
    return text

catalog_titles = sorted(eval_df["gold_title"].dropna().unique().tolist())
catalog_normalized_to_canonical = {normalize_title(t): t for t in catalog_titles}

print(f"Catalog size: {len(catalog_titles)}")
print("Sample titles:", catalog_titles[:5])


Catalog size: 100
Sample titles: ['12 Angry Men', '12th Fail', '2001: A Space Odyssey', 'A Clockwork Orange', 'Alien']


In [5]:
# Cell 5 - Load base model/tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()

print(f"Model loaded: {MODEL_ID}")
print(f"Device: {model.device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-7B-Instruct
Device: cuda:0


In [6]:
# Cell 6 - Standardized prompt template
catalog_block = "\n".join([f"- {t}" for t in catalog_titles])

SYSTEM_INSTRUCTION = (
    "You are a movie recommendation ranking engine. "
    "Return only strict JSON and no other text."
)

def build_user_prompt(description: str) -> str:
    return (
        "Given the movie description below, choose exactly 3 ranked movie titles from the allowed catalog.\n\n"
        "Requirements:\n"
        "1) Output exactly this JSON schema: {\"ranked_titles\":[\"title1\",\"title2\",\"title3\"]}\n"
        "2) Exactly 3 unique titles.\n"
        "3) Titles must come only from the allowed catalog.\n"
        "4) Rank best match first.\n"
        "5) No explanation, no markdown, no extra keys.\n\n"
        f"Description:\n{description}\n\n"
        f"Allowed catalog titles:\n{catalog_block}"
    )

def generate_raw_response(description: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": build_user_prompt(description)},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**model_inputs, **GEN_CONFIG)
    new_tokens = output_ids[0][len(model_inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [7]:
# Cell 7 - Inference loop (evaluation run)
records = []
for i, row in eval_df.iterrows():
    raw_text = generate_raw_response(row["prompt"])
    records.append(
        {
            "example_id": row["example_id"],
            "gold_title": row["gold_title"],
            "rank": row["Rank"],
            "prompt": row["prompt"],
            "raw_text": raw_text,
        }
    )
    if i < 3:
        print(f"Sample raw output {i + 1}: {raw_text}")
    if (i + 1) % 10 == 0:
        print(f"Generated {i + 1}/{len(eval_df)}")
pred_df = pd.DataFrame(records)
display(pred_df.head(2))
print(f"Inference complete. Rows: {len(pred_df)}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sample raw output 1: {"ranked_titles":["American Beauty","Forrest Gump","Cinema Paradiso"]}
Sample raw output 2: {"ranked_titles":["Inception","Interstellar","Eternal Sunshine of the Spotless Mind"]}
Sample raw output 3: {"ranked_titles":["2001: A Space Odyssey","Inception","Interstellar"]}
Generated 10/100
Generated 20/100
Generated 30/100
Generated 40/100
Generated 50/100
Generated 60/100
Generated 70/100
Generated 80/100
Generated 90/100
Generated 100/100


,example_id,gold_title,rank,prompt,raw_text
0,eval_0038,The Lion King,38,A young heir leaves home after a family traged...,"{""ranked_titles"":[""American Beauty"",""Forrest G..."
1,eval_0063,Princess Mononoke,63,A traveler tries to mediate between industrial...,"{""ranked_titles"":[""Inception"",""Interstellar"",""..."


Inference complete. Rows: 100


In [8]:
# Cell 8 - Parse + validate predictions
def parse_and_validate(raw_text: str):
    text = raw_text.strip()
    # 1) direct parse
    try:
        payload = json.loads(text)
    except Exception:
        payload = None
    # 2) fallback: extract first JSON object block
    if payload is None:
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            try:
                payload = json.loads(m.group(0))
            except Exception:
                payload = None
    if payload is None:
        return None, "invalid_json"
    if not isinstance(payload, dict) or "ranked_titles" not in payload:
        return None, "schema_error"
    ranked = payload["ranked_titles"]
    if not isinstance(ranked, list) or len(ranked) != TOP_K:
        return None, "schema_error"
    if any(not isinstance(x, str) for x in ranked):
        return None, "schema_error"
    normalized = [normalize_title(x) for x in ranked]
    if len(set(normalized)) != TOP_K:
        return None, "duplicate_titles"
    canonical = []
    for n in normalized:
        if n not in catalog_normalized_to_canonical:
            return None, "out_of_catalog"
        canonical.append(catalog_normalized_to_canonical[n])
    return canonical, "ok"
# Parser unit checks
ok_titles = catalog_titles[:3]
synthetic_ok = json.dumps({"ranked_titles": ok_titles})
synthetic_bad_json = "not-json"
synthetic_bad_len = json.dumps({"ranked_titles": ok_titles[:2]})
synthetic_dup = json.dumps({"ranked_titles": [ok_titles[0], ok_titles[0], ok_titles[2]]})
synthetic_ooc = json.dumps({"ranked_titles": ["Fake Movie", ok_titles[1], ok_titles[2]]})
assert parse_and_validate(synthetic_ok)[1] == "ok"
assert parse_and_validate(synthetic_bad_json)[1] == "invalid_json"
assert parse_and_validate(synthetic_bad_len)[1] == "schema_error"
assert parse_and_validate(synthetic_dup)[1] == "duplicate_titles"
assert parse_and_validate(synthetic_ooc)[1] == "out_of_catalog"
print("Parser unit checks passed.")
parsed_titles = []
parse_status = []
for raw in pred_df["raw_text"].tolist():
    parsed, status = parse_and_validate(raw)
    parsed_titles.append(parsed)
    parse_status.append(status)
pred_df["parsed_ranked_titles"] = parsed_titles
pred_df["parse_status"] = parse_status
pred_df["proposed_rank1"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
pred_df["proposed_rank2"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
pred_df["proposed_rank3"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[2] if isinstance(x, list) and len(x) > 2 else None)
display(pred_df[["example_id", "parse_status", "proposed_rank1", "proposed_rank2", "proposed_rank3"]].head(5))


Parser unit checks passed.


,example_id,parse_status,proposed_rank1,proposed_rank2,proposed_rank3
0,eval_0038,ok,American Beauty,Forrest Gump,Cinema Paradiso
1,eval_0063,ok,Inception,Interstellar,Eternal Sunshine of the Spotless Mind
2,eval_0084,ok,2001: A Space Odyssey,Inception,Interstellar
3,eval_0015,ok,Inception,Eternal Sunshine of the Spotless Mind,Fight Club
4,eval_0044,out_of_catalog,None,None,None


In [9]:
# Cell 9 - Reciprocal rank scoring
def reciprocal_rank(gold_title: str, ranked_titles):
    if not isinstance(ranked_titles, list):
        return 0.0
    for idx, title in enumerate(ranked_titles, start=1):
        if title == gold_title:
            return 1.0 / idx
    return 0.0

# Scoring unit checks
test_gold = "The Matrix"
assert reciprocal_rank(test_gold, ["The Matrix", "Inception", "Memento"]) == 1.0
assert reciprocal_rank(test_gold, ["Inception", "The Matrix", "Memento"]) == 0.5
assert reciprocal_rank(test_gold, ["Inception", "Memento", "The Matrix"]) == (1.0 / 3.0)
assert reciprocal_rank(test_gold, ["Inception", "Memento", "Interstellar"]) == 0.0
print("Scoring unit checks passed.")

pred_df["reciprocal_rank_pre"] = pred_df.apply(
    lambda r: reciprocal_rank(r["gold_title"], r["parsed_ranked_titles"]) if r["parse_status"] == "ok" else np.nan,
    axis=1,
)

valid_mask = pred_df["parse_status"] == "ok"
mrr_pre_adjudication = float(pred_df.loc[valid_mask, "reciprocal_rank_pre"].mean()) if valid_mask.any() else 0.0
print(f"Valid rows before adjudication: {valid_mask.sum()} / {len(pred_df)}")
print(f"MRR pre-adjudication (valid rows only): {mrr_pre_adjudication:.6f}")


Scoring unit checks passed.
Valid rows before adjudication: 70 / 100
MRR pre-adjudication (valid rows only): 0.242857


In [12]:
# Cell 10 - Adjudication artifacts (Colab-safe + backward-compatible)
queue_df = pred_df.loc[pred_df["parse_status"] != "ok", [
    "example_id",
    "gold_title",
    "raw_text",
    "parse_status",
    "proposed_rank1",
    "proposed_rank2",
    "proposed_rank3",
]].copy()

queue_df["notes"] = ""
queue_df.to_csv(ADJUDICATION_QUEUE_FILE, index=False)
print(f"Adjudication queue rows: {len(queue_df)} -> {ADJUDICATION_QUEUE_FILE}")

pred_df["final_ranked_titles"] = pred_df["parsed_ranked_titles"]

if ADJUDICATION_RESOLVED_FILE.exists():
    resolved = pd.read_csv(ADJUDICATION_RESOLVED_FILE)

    # Backward-compatible mapping: accept legacy queue-style columns
    if "resolution_status" not in resolved.columns and "parse_status" in resolved.columns:
        resolved["resolution_status"] = resolved["parse_status"].apply(
            lambda x: "accepted" if str(x).strip().lower() == "accepted" else "no_match"
        )
    if "rank1_title" not in resolved.columns and "proposed_rank1" in resolved.columns:
        resolved["rank1_title"] = resolved["proposed_rank1"]
    if "rank2_title" not in resolved.columns and "proposed_rank2" in resolved.columns:
        resolved["rank2_title"] = resolved["proposed_rank2"]
    if "rank3_title" not in resolved.columns and "proposed_rank3" in resolved.columns:
        resolved["rank3_title"] = resolved["proposed_rank3"]

    expected_cols = {"example_id", "resolution_status", "rank1_title", "rank2_title", "rank3_title"}
    missing_cols = expected_cols - set(resolved.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in resolved adjudication file: {sorted(missing_cols)}")

    for _, row in resolved.iterrows():
        if str(row["resolution_status"]).strip().lower() != "accepted":
            continue

        candidate = [row["rank1_title"], row["rank2_title"], row["rank3_title"]]
        candidate_norm = [normalize_title(x) for x in candidate]

        # Must be exactly 3 unique in-catalog titles
        if len(set(candidate_norm)) != TOP_K:
            continue
        if not all(n in catalog_normalized_to_canonical for n in candidate_norm):
            continue

        canonical = [catalog_normalized_to_canonical[n] for n in candidate_norm]

        # Safe scalar assignment per matching row
        match_idx = pred_df.index[pred_df["example_id"] == row["example_id"]].tolist()
        for idx in match_idx:
            pred_df.at[idx, "final_ranked_titles"] = canonical
else:
    print(f"No resolved adjudication file found at {ADJUDICATION_RESOLVED_FILE}. Using parsed outputs only.")

pred_df["reciprocal_rank_final"] = pred_df.apply(
    lambda r: reciprocal_rank(r["gold_title"], r["final_ranked_titles"]),
    axis=1,
)

mrr_final = float(pred_df["reciprocal_rank_final"].mean()) if len(pred_df) > 0 else 0.0
print(f"MRR final: {mrr_final:.6f}")


Adjudication queue rows: 30 -> outputs/baseline_adjudication_queue.csv
MRR final: 0.205000


In [13]:
# Cell 11 - Save outputs + summary
pred_records = []
for _, row in pred_df.iterrows():
    pred_records.append(
        {
            "example_id": row["example_id"],
            "gold_title": row["gold_title"],
            "raw_text": row["raw_text"],
            "parsed_ranked_titles": row["parsed_ranked_titles"] if isinstance(row["parsed_ranked_titles"], list) else None,
            "parse_status": row["parse_status"],
            "reciprocal_rank": float(row["reciprocal_rank_final"]),
        }
    )

with PREDICTIONS_FILE.open("w", encoding="utf-8") as f:
    for rec in pred_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

metrics = {
    "model_id": MODEL_ID,
    "eval_file": str(EVAL_FILE),
    "num_examples": int(len(pred_df)),
    "num_valid": int((pred_df["parse_status"] == "ok").sum()),
    "num_flagged_for_adjudication": int((pred_df["parse_status"] != "ok").sum()),
    "mrr_pre_adjudication": float(mrr_pre_adjudication),
    "mrr_final": float(mrr_final),
}

with METRICS_FILE.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# End-to-end smoke-check expectations when N_EVAL_LIMIT == 5
if N_EVAL_LIMIT == 5:
    assert PREDICTIONS_FILE.exists(), "Missing predictions output"
    assert ADJUDICATION_QUEUE_FILE.exists(), "Missing adjudication queue output"
    assert METRICS_FILE.exists(), "Missing metrics output"
    print("Smoke-check output assertions passed for 5-row run.")

summary_df = pd.DataFrame([metrics])
display(summary_df)
print(f"Saved predictions: {PREDICTIONS_FILE}")
print(f"Saved adjudication queue: {ADJUDICATION_QUEUE_FILE}")
print(f"Saved metrics: {METRICS_FILE}")


,model_id,eval_file,num_examples,num_valid,num_flagged_for_adjudication,mrr_pre_adjudication,mrr_final
0,Qwen/Qwen2.5-7B-Instruct,TestData.xlsx.csv,100,70,30,0.242857,0.205


Saved predictions: outputs/baseline_predictions.jsonl
Saved adjudication queue: outputs/baseline_adjudication_queue.csv
Saved metrics: outputs/baseline_metrics.json
